# Actually Compressing the Checkpoint File

**The problem:** `torch.save(model.state_dict(), path)` writes every tensor as 32-bit floats, regardless of how many distinct values it actually contains. Your ternary student's weights only take on 3 values ($-\alpha, 0, +\alpha$) per channel, but the saved file still spends a full 32 bits per weight -- which is why it comes out close to the same size as the FP32 baseline (both are ~11.17M params $\times$ 4 bytes $\approx$ 42.6 MB).

**What this notebook does:** writes a *custom* checkpoint format that actually packs each ternary weight into 2 bits (4 weights per byte), and stores everything else (the FP32-kept stem/FC/BatchNorm parameters, plus the per-channel $\alpha$ scales) at full precision as before. It then loads that compressed file back into a working model and confirms the accuracy is unchanged -- so you get a real, on-disk ~2.7 MB file, not just a number in a report.

**No GPU needed** -- this is pure serialization, not training.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = "/content/drive/MyDrive/atdl"
else:
    DRIVE_DIR = "."

STUDENT_CKPT = os.path.join(DRIVE_DIR, "ternary_resnet18_kd_qat_student_best.pth")
COMPRESSED_CKPT = os.path.join(DRIVE_DIR, "ternary_resnet18_student_COMPRESSED.pth")
print("Found." if os.path.exists(STUDENT_CKPT) else f"MISSING: {STUDENT_CKPT}")

Mounted at /content/drive
Found.


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Embedded: ternary quantizer + `TernaryResNet18`

In [3]:
def ternary_quantize(w, per_channel=True, delta_factor=0.7):
    orig_shape = w.shape
    if per_channel:
        w_flat = w.reshape(w.size(0), -1)
        delta = delta_factor * w_flat.abs().mean(dim=1, keepdim=True)
        mask = (w_flat.abs() > delta).float()
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        alpha = (w_flat.abs() * mask).sum(dim=1, keepdim=True) / denom
        w_q = (alpha * mask * torch.sign(w_flat)).reshape(orig_shape)
        return w_q, alpha.view(-1), delta.view(-1)
    else:
        delta = delta_factor * w.abs().mean()
        mask = (w.abs() > delta).float()
        denom = mask.sum().clamp(min=1.0)
        alpha = (w.abs() * mask).sum() / denom
        w_q = alpha * mask * torch.sign(w)
        return w_q, alpha, delta

class _TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, delta_factor, per_channel, clip_grad, clip_value):
        w_q, _, _ = ternary_quantize(w, per_channel=per_channel, delta_factor=delta_factor)
        return w_q
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None, None, None

def ternary_quantize_ste(w, delta_factor=0.7, per_channel=True, clip_grad=False, clip_value=1.0):
    return _TernarySTE.apply(w, delta_factor, per_channel, clip_grad, clip_value)

class TernaryConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=False,
                 delta_factor=0.7, per_channel=True, clip_grad=False, clip_value=1.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None
        self.stride, self.padding = stride, padding
        self.delta_factor, self.per_channel = delta_factor, per_channel
        self.clip_grad, self.clip_value = clip_grad, clip_value
        self.kernel_size, self.in_channels, self.out_channels = kernel_size, in_channels, out_channels
    def forward(self, x):
        w_q = ternary_quantize_ste(self.weight, self.delta_factor, self.per_channel, self.clip_grad, self.clip_value)
        return F.conv2d(x, w_q, self.bias, stride=self.stride, padding=self.padding)

class TernaryLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, delta_factor=0.7,
                 per_channel=True, clip_grad=False, clip_value=1.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.delta_factor, self.per_channel = delta_factor, per_channel
        self.clip_grad, self.clip_value = clip_grad, clip_value
        self.in_features, self.out_features = in_features, out_features
    def forward(self, x):
        w_q = ternary_quantize_ste(self.weight, self.delta_factor, self.per_channel, self.clip_grad, self.clip_value)
        return F.linear(x, w_q, self.bias)

class TernaryBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, ternary_kwargs=None):
        super().__init__()
        tk = ternary_kwargs or {}
        self.conv1 = TernaryConv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False, **tk)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = TernaryConv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False, **tk)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                TernaryConv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False, **tk),
                nn.BatchNorm2d(self.expansion * planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)

class TernaryResNetCIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10, ternary_kwargs=None, ternary_fc=False):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], 1, ternary_kwargs)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2, ternary_kwargs)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2, ternary_kwargs)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2, ternary_kwargs)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        if ternary_fc:
            self.fc = TernaryLinear(512 * block.expansion, num_classes, **(ternary_kwargs or {}))
        else:
            self.fc = nn.Linear(512 * block.expansion, num_classes)
    def _make_layer(self, block, planes, num_blocks, stride, ternary_kwargs):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s, ternary_kwargs=ternary_kwargs))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out); out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)

def TernaryResNet18(num_classes=10, delta_factor=0.7, per_channel=True, clip_grad=False, clip_value=1.0, ternary_fc=False):
    ternary_kwargs = dict(delta_factor=delta_factor, per_channel=per_channel, clip_grad=clip_grad, clip_value=clip_value)
    return TernaryResNetCIFAR(TernaryBasicBlock, [2, 2, 2, 2], num_classes, ternary_kwargs=ternary_kwargs, ternary_fc=ternary_fc)

## Load your existing (uncompressed, 42MB) checkpoint

In [4]:
student = TernaryResNet18(num_classes=10, delta_factor=0.7, per_channel=True, ternary_fc=False).to(DEVICE)
ckpt = torch.load(STUDENT_CKPT, map_location=DEVICE)
student.load_state_dict(ckpt["model_state_dict"])
student.eval()

uncompressed_size_mb = os.path.getsize(STUDENT_CKPT) / (1024 ** 2)
print(f"Existing checkpoint on disk: {uncompressed_size_mb:.2f} MB")

Existing checkpoint on disk: 42.70 MB


## Pack every ternary layer's weights into 2 bits each

Each ternary value is mapped to a 2-bit code (`0 -> 0b00`, `+alpha -> 0b01`, `-alpha -> 0b10`), then four codes are packed into a single byte using bit-shifts. `alpha` (one float per output channel) and every non-ternary parameter (stem conv, FC, BatchNorm, biases) are kept as ordinary float32 -- they're a tiny fraction of the total and packing them further isn't worth the complexity.

In [7]:
def pack_ternary_weight(w_q):
    """w_q: tensor with values in {-alpha_c, 0, +alpha_c} (already quantized). Returns packed uint8 bytes + shape."""
    flat = w_q.flatten()
    codes = torch.zeros_like(flat, dtype=torch.uint8)
    codes[flat > 0] = 1
    codes[flat < 0] = 2
    # pad to a multiple of 4 so we can pack 4 codes (2 bits each) per byte
    pad = (-len(codes)) % 4
    if pad:
        codes = torch.cat([codes, torch.zeros(pad, dtype=torch.uint8)])
    codes = codes.view(-1, 4)
    packed = (codes[:, 0] | (codes[:, 1] << 2) | (codes[:, 2] << 4) | (codes[:, 3] << 6)).to(torch.uint8)
    return packed, w_q.shape, pad


def unpack_ternary_weight(packed, shape, pad, alpha, per_channel):
    n_total = int(np.prod(shape))
    # Ensure 'codes' is created on the same device as 'packed'
    codes = torch.zeros(packed.numel() * 4, dtype=torch.uint8, device=packed.device)
    codes[0::4] = packed & 0b11
    codes[1::4] = (packed >> 2) & 0b11
    codes[2::4] = (packed >> 4) & 0b11
    codes[3::4] = (packed >> 6) & 0b11
    if pad:
        codes = codes[:-pad]
    codes = codes[:n_total].reshape(shape).float()

    sign = torch.zeros_like(codes)
    sign[codes == 1] = 1.0
    sign[codes == 2] = -1.0

    if per_channel:
        alpha_view = alpha.view(-1, *([1] * (len(shape) - 1)))
        return sign * alpha_view
    else:
        return sign * alpha


def save_compressed_checkpoint(model, path):
    compressed = {"other_params": {}, "ternary_layers": {}}
    for name, module in model.named_modules():
        if isinstance(module, (TernaryConv2d, TernaryLinear)):
            w = module.weight.detach()
            w_q, alpha, delta = ternary_quantize(w, per_channel=module.per_channel, delta_factor=module.delta_factor)
            packed, shape, pad = pack_ternary_weight(w_q)
            compressed["ternary_layers"][name] = {
                "packed": packed, "shape": shape, "pad": pad,
                "alpha": alpha.half(),  # float16 for the small alpha overhead too, extra savings
                "per_channel": module.per_channel,
                "bias": module.bias.detach().half() if module.bias is not None else None,
            }
    ternary_module_names = set(compressed["ternary_layers"].keys())
    for name, param in model.named_parameters():
        module_name = ".".join(name.split(".")[:-1])
        if module_name not in ternary_module_names:
            compressed["other_params"][name] = param.detach().clone()
    for name, buf in model.named_buffers():  # BatchNorm running mean/var
        compressed.setdefault("buffers", {})[name] = buf.detach().clone()

    torch.save(compressed, path)


save_compressed_checkpoint(student, COMPRESSED_CKPT)
compressed_size_mb = os.path.getsize(COMPRESSED_CKPT) / (1024 ** 2)
print(f"Compressed checkpoint on disk: {compressed_size_mb:.2f} MB")
print(f"Reduction vs. the original uncompressed .pth: {uncompressed_size_mb / compressed_size_mb:.1f}x")

Compressed checkpoint on disk: 2.82 MB
Reduction vs. the original uncompressed .pth: 15.2x


## Load the compressed file back into a real model and verify it still works

This is the important check: a smaller file is meaningless if it doesn't reconstruct the same model. Loads the compressed file, rebuilds a `TernaryResNet18`, and confirms its test accuracy exactly matches the original.

In [8]:
def load_compressed_checkpoint(path, num_classes=10):
    compressed = torch.load(path, map_location=DEVICE)
    model = TernaryResNet18(num_classes=num_classes, delta_factor=0.7, per_channel=True, ternary_fc=False).to(DEVICE)

    state_dict = model.state_dict()
    for name, tensor in compressed["other_params"].items():
        state_dict[name] = tensor
    for name, tensor in compressed.get("buffers", {}).items():
        state_dict[name] = tensor
    for name, layer_data in compressed["ternary_layers"].items():
        # Ensure alpha is on the correct device after conversion to float
        alpha_on_device = layer_data["alpha"].float().to(DEVICE)
        w = unpack_ternary_weight(layer_data["packed"], layer_data["shape"], layer_data["pad"],
                                   alpha_on_device, layer_data["per_channel"])
        state_dict[f"{name}.weight"] = w
        if layer_data["bias"] is not None:
            state_dict[f"{name}.bias"] = layer_data["bias"].float()

    model.load_state_dict(state_dict)
    model.eval()
    return model


reconstructed_student = load_compressed_checkpoint(COMPRESSED_CKPT)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=eval_transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

def evaluate_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total += x.size(0)
    return 100.0 * correct / total

original_acc = evaluate_acc(student, test_loader)
reconstructed_acc = evaluate_acc(reconstructed_student, test_loader)

print(f"Original (uncompressed) model test accuracy:   {original_acc:.2f}%")
print(f"Reconstructed (from compressed file) accuracy: {reconstructed_acc:.2f}%")
print(f"Match: {'YES -- compression is lossless for inference' if abs(original_acc - reconstructed_acc) < 0.01 else 'MISMATCH -- something is wrong, investigate!'}")

100%|██████████| 170M/170M [49:41<00:00, 57.2kB/s]


Original (uncompressed) model test accuracy:   95.15%
Reconstructed (from compressed file) accuracy: 95.15%
Match: YES -- compression is lossless for inference
